# Step 2. Cretae the retriever model

In [19]:
import os
import pickle
from langchain_core.documents import Document

from gensim import corpora
from gensim.parsing import strip_tags, strip_numeric, \
    strip_multiple_whitespaces, stem_text, strip_punctuation, \
    remove_stopwords, preprocess_string, strip_non_alphanum
from gensim import models
from gensim import similarities
import pymorphy3

from tqdm.notebook import tqdm

In [20]:
morph_analyzer = pymorphy3.MorphAnalyzer()

In [21]:
SPLITS_PATH = './splits'
GENSIM_PATH = './gensim'

In [22]:
with open(os.path.join(SPLITS_PATH, 'splits.pkl'), 'rb') as f:
    splits = pickle.load(f)
print(len(splits))

99


### Processing the splitted text

In [23]:
STOP_WORDS = ['и', 'в', 'во', 'не', 'что', 'он', 'на', 'я', 'с', 'со', 'как', 'а', 'то',
              'все', 'она', 'так', 'его', 'но', 'да', 'ты', 'к', 'у', 'же', 'вы', 'за',
              'бы', 'по', 'только', 'ее', 'мне', 'было', 'вот', 'от', 'меня', 'еще',
              'нет', 'о', 'из', 'ему', 'теперь', 'когда', 'даже', 'ну', 'вдруг', 'ли',
              'если', 'уже', 'или', 'ни', 'быть', 'был', 'него', 'до', 'вас', 'нибудь',
              'опять', 'уж', 'вам', 'ведь', 'там', 'потом', 'себя', 'ничего', 'ей',
              'может', 'они', 'тут', 'где', 'есть', 'надо', 'ней', 'для', 'мы', 'тебя',
              'их', 'чем', 'была', 'сам', 'чтоб', 'без', 'будто', 'чего', 'раз', 'тоже',
              'себе', 'под', 'будет', 'ж', 'тогда', 'кто', 'этот', 'того', 'потому',
              'этого', 'какой', 'совсем', 'ним', 'здесь', 'этом', 'один', 'почти', 'мой',
              'тем', 'чтобы', 'нее', 'сейчас', 'были', 'куда', 'зачем', 'всех', 'никогда',
              'можно', 'при', 'наконец', 'два', 'об', 'другой', 'хоть', 'после', 'над',
              'больше', 'тот', 'через', 'эти', 'нас', 'про', 'всего', 'них', 'какая',
              'много', 'разве', 'три', 'эту', 'моя', 'впрочем', 'хорошо', 'свою', 'этой',
              'перед', 'иногда', 'лучше', 'чуть', 'том', 'нельзя', 'такой', 'им', 'более',
              'всегда', 'конечно', 'всю', 'между']

In [24]:
# Filters to be executed in pipeline
transform_to_lower = lambda s: s.lower()
CLEAN_FILTERS = [
                strip_tags,
                # strip_numeric,
                strip_punctuation,
                strip_non_alphanum,
                strip_multiple_whitespaces,
                transform_to_lower,
                ]

In [25]:
# Method does the filtering of all the unrelevant text elements
def cleaning_pipe(text:str) -> list[str]:
    # Invoking gensim.parsing.preprocess_string method with set of filters
    processed_words = preprocess_string(text, CLEAN_FILTERS)
    processed_words = [s for s in processed_words if len(s) > 1]
    processed_words = [s for s in processed_words if s not in STOP_WORDS]
    processed_words = [morph_analyzer.parse(s)[0].normal_form for s in processed_words]
    return processed_words

In [26]:
morph_analyzer.parse('Страннике')[0].normal_form

'странник'

In [27]:
%%time
processed_docs = []
for doc in tqdm(splits):
    processed_docs.append(cleaning_pipe(doc.page_content))

  0%|          | 0/99 [00:00<?, ?it/s]

CPU times: total: 2.97 s
Wall time: 2.97 s


In [28]:
processed_docs[10][:10]

['июнь',
 '78',
 'го',
 'год',
 'общий',
 'этот',
 'переписка',
 'оставить',
 'какой',
 'тягостный']

### Create Dictionary, Model, Index

In [29]:
%%time
dictionary = corpora.Dictionary(processed_docs)
with open(os.path.join(GENSIM_PATH, 'dictionary.pkl'), 'wb') as h:
    pickle.dump(dictionary, h)
print(dictionary)

Dictionary<7492 unique tokens: ['дверь', 'зверь', 'маленький', 'мальчик', 'около']...>
CPU times: total: 31.2 ms
Wall time: 35 ms


In [30]:
%%time
bow_corpus = [dictionary.doc2bow(text) for text in processed_docs]
model = models.TfidfModel(bow_corpus)
model.save(os.path.join(GENSIM_PATH, 'tfidf_model.mo'))

CPU times: total: 46.9 ms
Wall time: 51 ms


In [31]:
%%time
index = similarities.SparseMatrixSimilarity(model[bow_corpus], num_features=len(dictionary))
with open(os.path.join(GENSIM_PATH, 'similarity_index.pkl'), 'wb') as h:
    pickle.dump(index, h)

CPU times: total: 31.2 ms
Wall time: 41 ms


### Search engine and test

In [32]:
def get_closest_n(query, n):
    '''get the top matching docs as per cosine similarity
    between tfidf vector of query and all docs'''
    query_document = cleaning_pipe(query)
    query_bow = dictionary.doc2bow(query_document)
    sims = index[model[query_bow]]
    top_idx = sims.argsort()[-1*n:][::-1]
    
    result = []
    for idx in top_idx:
        similarity = round(float(sims[idx]), 3)
        doc = splits[idx]
        result.append((doc, similarity))
    return result

In [33]:
query = """Что напомнило Бромбергу о саркофаге в связи с Александром Дымком?"""

In [34]:
# Check the result
result = get_closest_n(query, n=3)
for doc, similarity in result:
    for k,v in doc.metadata.items():
        if k in ['chapter', 'paragraph', 'section', 'size']:
            print(f'{k}: {v}')
    # print(f'Context: {doc.page_content[:500]}...')
    print(f'similarity: {similarity}\n')

chapter: 4 июня 78–го года
section: Сегодня а вернее вчера...
size: 4009
similarity: 0.343

chapter: 3 июня 78–го года
section: ЗАСТАВА НА РЕКЕ ТЕЛОН
size: 5302
similarity: 0.166

chapter: 4 июня 78–го года
section: ЛЕВ АБАЛКИН У ДОКТОРА БРОМБЕРГА
size: 4166
similarity: 0.139



In [35]:
query = """Кирилл Александров, известный своими антропоморфистскими взглядами, высказал предположение, что саркофаг есть хранилище генофонда Странников. Все известные мне доказательства негуманоидности Странников, заявил он, являются по сути своей косвенными. На самом же деле Странники вполне могут оказаться генетическими двойниками человека. Такое предположение не противоречит ни одному из доступных фактов. Исходя из этого, Александров предлагал все исследования прекратить, вернуть находку в первоначальное состояние и покинуть систему ЕН 9173."""

In [36]:
result = get_closest_n(query, n=3)
for doc, similarity in result:
    for k,v in doc.metadata.items():
        if k in ['chapter', 'paragraph', 'section', 'size']:
            print(f'{k}: {v}')
    # print(f'Context: {doc.page_content[:500]}...')
    print(f'similarity: {similarity}\n')

chapter: 4 июня 78–го года
section: Кирилл Александров...
size: 4019
similarity: 0.429

chapter: 4 июня 78–го года
section: Что же касается Геннадия...
size: 4179
similarity: 0.086

chapter: 4 июня 78–го года
section: ТАЙНА ЛИЧНОСТИ ЛЬВА АБАЛКИНА
size: 3450
similarity: 0.069

